### Step 1: Use annotated max heights to estimate how repo height varies with repo area

In [1]:
import pandas as pd
import numpy as np
import os
import pyproj

In [2]:
# make save directory
out_dir = "../out/20240905/total_capacity/"
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

# load in flux file first
raw_data_root = "/Users/nick/Projects/gangaSandFlux/dat/20240905/"
filename = "GangesStockpiles_basal_area_points_final.csv"
sand_repo_df = pd.read_csv(os.path.join(raw_data_root, filename))
sand_repo_df = sand_repo_df.iloc[:, :-2].dropna(inplace=False)
sand_repo_df.reset_index(inplace=True, drop=True)

sand_repo_df.shape

(411410, 6)

In [ ]:
old_data = pd.read_csv("/Users/nick/Projects/gangaSandFlux/dat/20220214/sbapts_02142022putm.csv")
date_vec = old_data.loc[:, "Date"].tolist()
year_vec = [int(d[-4:]) for d in date_vec]
sand_repo_df["year"] = year_vec 
old_stats = old_data.loc[""]

In [3]:
# convert repo coordinates from lat lon to meters
projecter = pyproj.Proj(proj='utm', zone=45, ellps='WGS84', preserve_units=True)
point_tuple = projecter(sand_repo_df.loc[:, "POINT_X"], sand_repo_df.loc[:, "POINT_Y"])

# add to DF
sand_repo_df.loc[:, "point_x_m"] = point_tuple[0]-np.min(point_tuple[0])
sand_repo_df.loc[:, "point_y_m"] = point_tuple[1]-np.min(point_tuple[1])

sand_repo_df.loc[:, "AreaName"] = sand_repo_df.loc[:, "AreaName"].astype(str)
sand_repo_df.loc[:, "numID"] = sand_repo_df.loc[:, "numID"].astype(int)
sand_repo_df.loc[:, "Date"] = sand_repo_df.loc[:, "Date"].astype(str)
name_vec = sand_repo_df.loc[:, "Name"].tolist()
year_vec = [int(d[-4:]) for d in date_vec]
sand_repo_df["year"] = year_vec 
# sand_repo_df.loc[:, "Type"] = sand_repo_df.loc[:, "Type"].astype(str)
# sand_flux_df = sand_flux_df.loc[sand_flux_df["Type"] != "nan"].reset_index(inplace=True, drop=True)


# make new ID variable
id_vec = [sand_repo_df.loc[p, "AreaName"] + "_" + sand_repo_df.loc[p, "Date"] + "_" 
                 + str(sand_repo_df.loc[p, "numID"]) for p in range(sand_repo_df.shape[0])] 
sand_repo_df.loc[:, "repoID"] = id_vec

/var/folders/m7/tpjttxb92tl8b4c9svgvgby00000gn/T/ipykernel_38671/1877629667.py:10: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  sand_repo_df.loc[:, "numID"] = sand_repo_df.loc[:, "numID"].astype(int)


In [7]:
import plotly.express as px
sand_repo_df["year_str"] = sand_repo_df["year"].astype("str")
indices = np.random.choice(range(sand_repo_df.shape[0]), 25000, replace=False)

fig = px.scatter(sand_repo_df.loc[indices], x="point_x_m", y="point_y_m", color="year_str")
fig.show()

In [15]:
sand_repo_df.loc[:, ["year", "repoID"]].groupby(by="year").count()

,repoID
year,
2001,26
2004,52
2006,17
2007,5860
2008,360
2009,5506
2010,5175
2011,32383
2012,5419


In [9]:
def PolyArea(x,y):
    return 0.5*np.abs(np.dot(x,np.roll(y,1))-np.dot(y,np.roll(x,1)))

### Calculate repo base areas
This takes a few minuts to run

In [12]:
from tqdm import tqdm
import pandas as pd
from shapely.geometry import Polygon

# iterate through repos and calculate mh distances
repo_index = np.unique(sand_repo_df["repoID"])
repo_dim_vec = []
repo_area_vec = []
repo_date_vec = []
repo_year_vec = []
repo_region_vec = []
repo_id_vec = []
repo_string_vec = []

for p, point_name in enumerate(tqdm(repo_index)):
    
    # filter
    df_temp = sand_repo_df.loc[sand_repo_df["repoID"]==point_name, :]
    df_temp = df_temp.reset_index(drop=True)
    base_points_xy = df_temp.loc[:, ["point_x_m", "point_y_m"]].to_numpy()
    
    if df_temp.shape[0] > 4 and np.all(~np.isinf(base_points_xy)) and np.all(~np.isnan(base_points_xy)): 
        
#         value = [tuple(base_points_xy[i, :]) for i in range(base_points_xy.shape[0])]
#         polygon =                               
        # calculate area
#         area = polygon.area
        area = PolyArea(base_points_xy[:,0], base_points_xy[:,1]) 
        length = np.sqrt(area)
        year = df_temp.loc[0, "year"]
        date = df_temp.loc[0, "Date"]
        split_out = point_name.split("_")
        name = split_out[0]
        instance = split_out[-1]
        
        # save
        repo_string_vec.append(point_name)
        repo_area_vec.append(area)
        repo_dim_vec.append(length)
        repo_date_vec.append(date)
        repo_year_vec.append(year)
        repo_region_vec.append(name)
        repo_id_vec.append(instance)
        

repo_df = pd.DataFrame(repo_string_vec, columns=["repoID"])
repo_df["region"] = repo_region_vec
repo_df["id"] = repo_id_vec
repo_df["date"] = repo_date_vec
repo_df["year"] = repo_year_vec
repo_df["area_m2"] = repo_area_vec
repo_df["side_dim_m"] = repo_dim_vec

100%|██████████| 29635/29635 [05:41<00:00, 86.75it/s]


#### Use bootstrap resampling to estimate the mean and standard error of the max height as a function of repo linear dimension

In [14]:
import plotly.express as px

stat_df = repo_df.loc[:, ["year", "area_m2", "side_dim_m"]].groupby(by="year").count()
stat_df.reset_index(inplace=True)

fig = px.bar(stat_df, x="year", y="area_m2")
fig.update_layout(yaxis_title="total repo area (m^2)")
fig.show()

In [38]:
stat_df = repo_df.loc[:, ["year", "area_m2", "region", "side_dim_m"]].groupby(by=["region", "year"]).sum()
stat_df.reset_index(inplace=True)

fig = px.bar(stat_df, x="year", y="area_m2", color="region")
fig.update_layout(yaxis_title="mean repo size (m^2)",)
fig.show()

In [6]:
import scipy

repo_areas = np.asarray(repo_area_vec)
repo_heights = np.asarray(mh_length_vec) * np.tan(np.pi/6)
# fit height function
# [max_h, area_hm, n]
x0 = [9, 2e3, 1]
# max_h = np.percentile(repo_heights, 99)
param_fit_vec = []
n_boots = 100
boot_options = range(len(repo_areas))
repo_lengths = np.sqrt(repo_areas)

for n in tqdm(range(n_boots), "Performing bootstrap fits..."):
    boot_indices = np.random.choice(boot_options, len(boot_options), replace=True)
    
    def height_fun(params, repo_areas=repo_lengths[boot_indices]):
        h = np.divide(params[0] * repo_areas**params[2], (params[1]**params[2] + repo_areas**params[2]))
        return h

    def loss_fun(params, repo_heights=repo_heights[boot_indices]):
        h_hat = height_fun(params)
        return repo_heights - h_hat


    params_fit = scipy.optimize.least_squares(loss_fun, x0, bounds=[(0, 0, 0), (np.inf, np.inf, np.inf)])
    
    param_fit_vec.append(params_fit)
    

# get full fit on all paramters
def height_fun(params, repo_areas=repo_lengths):
        h = np.divide(params[0] * repo_areas**params[2], (params[1]**params[2] + repo_areas**params[2]))
        return h

def loss_fun(params, repo_heights=repo_heights):
    h_hat = height_fun(params)
    return repo_heights - h_hat

param_fit_full = scipy.optimize.least_squares(loss_fun, x0, bounds=[(0, 0, 0), (np.inf, np.inf, np.inf)])

Performing bootstrap fits...: 100%|██████████| 100/100 [00:01<00:00, 59.34it/s]


### Calculate the starndard error and mean as a function of length
Use this as a basis for estimating the standard deviation of the data about the inferred mean. We will use this for sampling

In [10]:
import plotly.graph_objects as go 
import plotly.express as px
from scipy.spatial import distance_matrix

length_vec = np.linspace(0, 500, 1000)
height_trend_array = np.empty((n_boots, len(length_vec)))

dists = distance_matrix(repo_lengths[:, np.newaxis], length_vec[:, np.newaxis])
l_indices = np.argmin(dists, axis=1)

for n in range(n_boots):
    height_trend_array[n, :] = height_fun(param_fit_vec[n].x, repo_areas=length_vec)

height_mean = height_fun(param_fit_vec[n].x, repo_areas=length_vec)
height_mean[0] = 0 # np.mean(height_trend_array, axis=0)
height_se = np.std(height_trend_array, axis=0)
ub = height_mean + height_se
lb = height_mean - height_se

# numerically estimage SD by expanding SE by constant until 68.27% of data fall within band
increment = 0.01
factor = 1
height_sigma = height_se
frac_inside = 0
while frac_inside < .6827:
    factor += increment
#     height_sigma *= factor
    ubs = height_mean + height_sigma*factor
    lbs = height_mean - height_sigma*factor
    frac_inside = np.mean((repo_heights <= ubs[l_indices]) & (repo_heights >= lbs[l_indices]))
    
fig = px.scatter(x=np.sqrt(repo_areas), y=repo_heights)
fig.add_trace(go.Scatter(x=length_vec, y=height_mean, mode="lines"))
fig.add_trace(go.Scatter(
        x=np.c_[length_vec, length_vec[::-1]].T.ravel(), 
        y=np.c_[ubs, lbs[::-1]].T.ravel(), # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(0,0,0,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ))
fig.add_trace(go.Scatter(
        x=np.c_[length_vec, length_vec[::-1]].T.ravel(), 
        y=np.c_[ub, lb[::-1]].T.ravel(), # upper, then lower reversed
        fill='toself',
        fillcolor='rgba(0,0,0,0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        hoverinfo="skip",
        showlegend=False
    ))
fig.show()

### Save

In [9]:
import pickle

# Write the fit object to file
with open(os.path.join(out_dir, 'max_height_fit.pkl'), 'wb') as f:
    pickle.dump(param_fit_full, f)

In [15]:
import pandas as pd

# make lookup table
height_ref_df = pd.DataFrame(length_vec, columns=["side_dim_m"])
height_ref_df["max_height_m"] = height_mean
height_ref_df["height_se"] = height_se
height_ref_df["height_sd"] = height_se*factor
height_ref_df["overall_max_m"] = np.max(repo_heights)

height_ref_df.to_csv(os.path.join(out_dir, 'height_df.csv'), index=False)